# 06b — Locked test: one evaluation of the frozen models

Applies the models and thresholds frozen by **Notebook 06a** (`operating_policy.json`, internal validation only)
**once** to the official MURA `valid` split, XR_WRIST. Nothing is tuned here, and the notebook refuses to run if a
locked-test result already exists.

Run it only after the team has frozen the model; afterwards the test split no longer gives an unbiased estimate
for any changed model.

**Disclosure.** The official MURA `valid` split was already observed once in Phase 1 for a frozen reference model
(`reports/TEST_SET_GOVERNANCE.md`); it is reported as a *held-out test previously observed in Phase 1*.

Outputs: `DATA/RUNS_ARTEM/06B_LOCKED_TEST/run_…/`. Inputs are read-only.

In [ ]:
import hashlib
import json
import random
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
    roc_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| Torch:", torch.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# The team DATA folder is shared from the project owner's Drive. Members see it
# either through a "DATA" shortcut in My Drive or through the shared
# "Final Project DS23" shortcut. Both resolve to the same physical folder.
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR_CANDIDATES = [
    DRIVE_ROOT / "DATA",
    DRIVE_ROOT
    / "Final Project DS23"
    / "Agentic AI-Assisted Detection of Musculoskeletal Abnormalities in Emergency X-rays"
    / "DATA",
]


def resolve_data_dir(candidates=DATA_DIR_CANDIDATES):
    for candidate in candidates:
        if (candidate / "MURA-v1.1_files.zip").exists() and (candidate / "WRIST_SPLITS").is_dir():
            return candidate
    raise FileNotFoundError(
        "Team DATA folder not found. Add a 'DATA' shortcut to My Drive or check "
        "that 'Final Project DS23' is in My Drive. Tried:\n"
        + "\n".join(str(c) for c in candidates)
    )


DATA_DIR = resolve_data_dir()
RAW_ZIP = DATA_DIR / "MURA-v1.1_files.zip"
SPLITS_DIR = DATA_DIR / "WRIST_SPLITS"            # read-only here (Notebook 01 output)
PREPROCESSING_DIR = DATA_DIR / "PREPROCESSED_WRIST"  # read-only here (Notebook 02 output)
MODEL_A_DIR = DATA_DIR / "BASELINE_MODEL_A"        # read-only here (Notebook 03 output)

# Shared-folder rule: this notebook never modifies or replaces existing team
# files. Everything it writes goes to a NEW run folder under RUNS_ARTEM.
RUN_ROOT = DATA_DIR / "RUNS_ARTEM"
STAGE_DIR = RUN_ROOT / "06B_LOCKED_TEST"


def new_run_dir(stage_dir):
    run_id = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
    run_dir = stage_dir / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def out_path(run_dir, name):
    """Return a path inside this run's own folder; refuse anything else."""
    path = run_dir / name
    assert str(path).startswith(str(run_dir) + "/"), path
    assert not path.exists(), f"Refusing to overwrite: {path}"
    return path


print("Team DATA folder:", DATA_DIR)
print("Writes only under:", STAGE_DIR)

## Settings and frozen policy

In [ ]:
POLICY_RUN_ID = "SET_ME"  # printed by Notebook 06a, e.g. "run_20261017T090000Z"
BOOTSTRAP_ITERATIONS = 1000

POLICY_PATH = RUN_ROOT / "06A_OPERATING_POLICY" / POLICY_RUN_ID / "operating_policy.json"
assert POLICY_PATH.exists(), f"Frozen policy not found: {POLICY_PATH}"

# One look at the test split: refuse to run if any locked-test result already exists.
previous = []
for stage in ("06B_LOCKED_TEST", "06_POLICY_AND_LOCKED_TEST"):
    if (RUN_ROOT / stage).exists():
        previous += sorted((RUN_ROOT / stage).glob("run_*/locked_test_metrics.json"))
assert not previous, (
    "A locked-test evaluation already exists; the test split must not be evaluated again:\n"
    + "\n".join(map(str, previous))
)


def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


POLICY_SHA256 = sha256_of(POLICY_PATH)
policy = json.loads(POLICY_PATH.read_text())
assert policy["official_mura_valid_read"] is False
PRIMARY_SENSITIVITY_TARGET = policy["primary_sensitivity_target"]

# Paths in the policy are relative to the team DATA folder (older policies stored absolute
# paths; DATA_DIR / absolute_path returns the absolute path, so both forms work).
MODEL_A_FILE = DATA_DIR / policy["models"]["A"]["artifact"]
MODEL_B_FILE = DATA_DIR / policy["models"]["B"]["artifact"]
for name in ("A", "B"):
    frozen = policy["models"][name]
    assert sha256_of(DATA_DIR / frozen["artifact"]) == frozen["artifact_sha256"], f"Model {name} changed after freezing"
    assert sha256_of(DATA_DIR / frozen["validation_predictions"]) == frozen["validation_predictions_sha256"], name
val_a = pd.read_csv(DATA_DIR / policy["models"]["A"]["validation_predictions"])
val_b = pd.read_csv(DATA_DIR / policy["models"]["B"]["validation_predictions"])

RUN_DIR = new_run_dir(STAGE_DIR)
print("Frozen policy:", POLICY_RUN_ID, "| sha256:", POLICY_SHA256[:12])
print("Model B run:", policy["models"]["B"]["run_id"], "| pipeline model:", policy["pipeline_model"])
print("This run writes to:", RUN_DIR)

In [ ]:
def classification_metrics(y_true, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "threshold": float(threshold),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "flagged_abnormal": int(predicted.sum()),
        "flagged_percent": float(100 * predicted.mean()),
    }


def highest_threshold_reaching(y_true, probabilities, target):
    # Same rule as Notebook 04: highest validation threshold with sensitivity >= target.
    for threshold in np.unique(probabilities)[::-1]:
        if classification_metrics(y_true, probabilities, threshold)["sensitivity"] >= target:
            return float(threshold)
    raise RuntimeError(f"No threshold reaches sensitivity {target}")

## Test manifest (official MURA `valid`, XR_WRIST)

In [ ]:
TEST_MANIFEST = SPLITS_DIR / "wrist_official_validation.csv"
TEST_STUDIES = SPLITS_DIR / "wrist_official_validation_studies.csv"
test_views = pd.read_csv(TEST_MANIFEST)

PATH_CANDIDATES = ("image_path", "relative_path", "path", "file_name")
PATH_COLUMN = next((c for c in PATH_CANDIDATES if c in test_views.columns), None)
assert PATH_COLUMN is not None


def to_binary_label(values):
    # Same idea as Notebook 02: numeric 0/1 or MURA text labels.
    text = values.astype(str).str.strip().str.lower()
    mapping = {"0": 0, "1": 1, "0.0": 0, "1.0": 1, "negative": 0, "normal": 0, "positive": 1, "abnormal": 1}
    mapped = text.map(mapping)
    unknown = sorted(text[mapped.isna()].unique())
    assert not unknown, f"Unsupported labels: {unknown}"
    return mapped.astype(int)


# Notebook 01 saves the test manifest with the 5 index columns only; patient, study and label
# are encoded in the MURA path (<split>/XR_WRIST/patient<N>/study<N>_<label>/image<N>.png).
names = test_views[PATH_COLUMN].astype(str).str.replace("\\", "/", regex=False)
parsed = names.str.extract(r"(?P<patient_id>patient\d+)/(?P<study_id>study\d+_(?P<label_text>positive|negative))/")
for column in ("patient_id", "study_id"):
    if column not in test_views.columns:
        test_views[column] = parsed[column]
if "study_uid" not in test_views.columns:
    test_views["study_uid"] = parsed["patient_id"] + "/" + parsed["study_id"]
label_source = test_views["label"] if "label" in test_views.columns else parsed["label_text"]
test_views["label_binary"] = to_binary_label(label_source)
assert test_views[["patient_id", "study_uid"]].notna().all().all(), "Unparsed test paths"

apple_double = names.str.split("/").str[-1].str.startswith("._")
print("AppleDouble rows removed:", int(apple_double.sum()))
test_views = test_views.loc[~apple_double].reset_index(drop=True)
names = names.loc[~apple_double].reset_index(drop=True)
assert names.str.contains(r"(?:^|/)valid/", regex=True).all(), "Test manifest contains non-valid images"
assert test_views.groupby("study_uid")["label_binary"].nunique().max() == 1

# Cross-check every test study label with the official study-label table (study_path, label 0/1).
official = pd.read_csv(TEST_STUDIES)
official_uid = official["study_path"].astype(str).str.extract(r"(patient\d+/study\d+_(?:positive|negative))")[0]
official_labels = dict(zip(official_uid, official["label"].astype(int)))
mine = test_views.drop_duplicates("study_uid").set_index("study_uid")["label_binary"]
assert set(mine.index) == set(official_labels), "Test studies differ from the official study-label table"
assert all(official_labels[s] == l for s, l in mine.items()), "Study label mismatch with the official table"

# Known issue (Notebook 01, Step 14): one internal-train image is byte-identical to one test image.
KNOWN_DUPLICATE_TEST_STUDY = "patient11359/study1_negative"

train_manifest = pd.read_csv(PREPROCESSING_DIR / "wrist_internal_train_clean.csv")
validation_manifest = pd.read_csv(PREPROCESSING_DIR / "wrist_internal_val_clean.csv")
development_patients = set(train_manifest["patient_id"].astype(str)) | set(validation_manifest["patient_id"].astype(str))
overlap = development_patients & set(test_views["patient_id"].astype(str))
print("Patients shared between development and test:", len(overlap))
studies = test_views.drop_duplicates("study_uid")
print(f"Test: {len(test_views)} images, {len(studies)} studies, {studies['patient_id'].nunique()} patients, "
      f"{int(studies['label_binary'].sum())} abnormal")

In [ ]:
from PIL import Image, ImageOps
from torchvision import transforms
from torchvision.transforms import InterpolationMode


class PadToSquare:
    """Same padding rule as Notebooks 02 and 03 (black border, centred)."""

    def __call__(self, image):
        width, height = image.size
        side = max(width, height)
        left = (side - width) // 2
        top = (side - height) // 2
        right = side - width - left
        bottom = side - height - top
        return ImageOps.expand(image, border=(left, top, right, bottom), fill=0)


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGE_SIZE = 224


def _to_gray(image):
    return image.convert("L")


def _to_rgb(image):
    return image.convert("RGB")


# Identical to the evaluation transform of Notebook 03.
evaluation_transform = transforms.Compose([
    transforms.Lambda(_to_gray),
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    transforms.Lambda(_to_rgb),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
import shutil
import time
import zipfile

LOCAL_ROOT = Path("/content/mura_local")
LOCAL_ZIP = LOCAL_ROOT / "MURA-v1.1_files.zip"
LOCAL_IMAGES = LOCAL_ROOT / "images"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)


def stage_zip_locally():
    """Copy the 3.4 GB archive from Drive to the Colab disk once (read-only use)."""
    if LOCAL_ZIP.exists() and LOCAL_ZIP.stat().st_size == RAW_ZIP.stat().st_size:
        print("Local ZIP already present.")
        return
    started = time.time()
    shutil.copyfile(RAW_ZIP, LOCAL_ZIP)
    assert LOCAL_ZIP.stat().st_size == RAW_ZIP.stat().st_size
    print(f"Copied ZIP to local disk in {(time.time() - started) / 60:.1f} min")


def resolve_zip_member(value, members):
    normalized = str(value).replace("\\", "/").lstrip("/")
    for candidate in (normalized, "MURA-v1.1/" + normalized):
        if candidate in members:
            return candidate
    raise KeyError(f"Image path not found in ZIP: {value}")


def extract_members(paths):
    """Extract the listed images to local disk; return {manifest path: local file}."""
    with zipfile.ZipFile(LOCAL_ZIP) as archive:
        members = set(archive.namelist())
        mapping = {}
        for value in paths:
            member = resolve_zip_member(value, members)
            target = LOCAL_IMAGES / member
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, open(target, "wb") as sink:
                    shutil.copyfileobj(source, sink)
            mapping[value] = target
    return mapping

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset


def load_image(path):
    with Image.open(path) as image:
        image.load()
        return image.copy()


class StudyDataset(Dataset):
    """One item per study: all its views stacked as [V, 3, 224, 224]."""

    def __init__(self, views, local_paths, transform, path_column):
        self.transform = transform
        self.studies = []
        for study_uid, group in views.groupby("study_uid", sort=True):
            assert group["patient_id"].nunique() == 1
            assert group["label_binary"].nunique() == 1
            self.studies.append({
                "study_id": str(study_uid),
                "patient_id": str(group["patient_id"].iloc[0]),
                "label": int(group["label_binary"].iloc[0]),
                "view_paths": [str(p) for p in group[path_column]],
                "local_paths": [local_paths[p] for p in group[path_column]],
            })

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, index):
        study = self.studies[index]
        images = torch.stack([self.transform(load_image(p)) for p in study["local_paths"]])
        return {
            "images": images,
            "label": float(study["label"]),
            "study_id": study["study_id"],
            "patient_id": study["patient_id"],
            "index": index,
        }


def collate_studies(batch):
    max_views = max(item["images"].shape[0] for item in batch)
    _, channels, height, width = batch[0]["images"].shape
    images = torch.zeros((len(batch), max_views, channels, height, width))
    mask = torch.zeros((len(batch), max_views), dtype=torch.bool)
    for row, item in enumerate(batch):
        views = item["images"].shape[0]
        images[row, :views] = item["images"]
        mask[row, :views] = True
    return {
        "images": images,
        "mask": mask,
        "labels": torch.tensor([item["label"] for item in batch], dtype=torch.float32),
        "study_ids": [item["study_id"] for item in batch],
        "patient_ids": [item["patient_id"] for item in batch],
        "indices": [item["index"] for item in batch],
    }

In [ ]:
import torch.nn as nn
from torchvision import models


def build_model_b(pretrained=True):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)  # one logit per radiograph
    return model


def study_forward(model, images, mask, use_amp):
    """Per-view logits -> probabilities -> mean and max per study (padding ignored)."""
    flat = images[mask]
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
        logits = model(flat).squeeze(1)
    view_probs_flat = torch.sigmoid(logits.float())
    view_probs = torch.zeros(mask.shape, device=images.device)
    view_probs[mask] = view_probs_flat
    counts = mask.sum(dim=1).clamp(min=1)
    mean_prob = (view_probs * mask).sum(dim=1) / counts
    max_prob = view_probs.masked_fill(~mask, -1.0).max(dim=1).values
    return mean_prob, max_prob, view_probs

In [ ]:
stage_zip_locally()
local_paths = extract_members(test_views[PATH_COLUMN].tolist())

# Full decode check (as in Notebook 02); unreadable files are reported and excluded.
unreadable = []
for value, local in local_paths.items():
    try:
        with Image.open(local) as image:
            image.load()
    except Exception as error:  # noqa: BLE001
        unreadable.append({"image_path": value, "reason": str(error) or type(error).__name__})
if unreadable:
    display(pd.DataFrame(unreadable))
    bad = {row["image_path"] for row in unreadable}
    test_views = test_views.loc[~test_views[PATH_COLUMN].isin(bad)].reset_index(drop=True)
print("Unreadable test images:", len(unreadable))

from torch.utils.data import DataLoader

test_dataset = StudyDataset(test_views, local_paths, evaluation_transform, PATH_COLUMN)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2, collate_fn=collate_studies)
print("Test studies:", len(test_dataset))

## Model A on the test split (frozen encoder + saved Logistic Regression)

In [ ]:
import torch.nn as nn
from torchvision import models

encoder = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
encoder.fc = nn.Identity()
encoder = encoder.to(DEVICE).eval()
assert sha256_of(MODEL_A_FILE) == policy["models"]["A"]["artifact_sha256"]
model_a = joblib.load(MODEL_A_FILE)


@torch.no_grad()
def predict_model_a(loader):
    rows = []
    for batch in loader:
        images, mask = batch["images"].to(DEVICE), batch["mask"].to(DEVICE)
        embeddings = encoder(images[mask]).float().cpu().numpy()
        counts = mask.sum(dim=1).cpu().numpy()
        start = 0
        for i, index in enumerate(batch["indices"]):
            study = loader.dataset.studies[index]
            pooled = embeddings[start:start + counts[i]].mean(axis=0, keepdims=True)
            start += counts[i]
            rows.append({
                "study_id": study["study_id"],
                "patient_id": study["patient_id"],
                "label": study["label"],
                "num_views": int(counts[i]),
                "abnormal_probability": float(model_a.predict_proba(pooled)[0, 1]),
            })
    return pd.DataFrame(rows)


print("Model A ready (frozen encoder + saved Logistic Regression).")

## Model B on the test split (frozen checkpoint)

In [ ]:
model_b = build_model_b(pretrained=False).to(DEVICE)
assert sha256_of(MODEL_B_FILE) == policy["models"]["B"]["artifact_sha256"]
model_b.load_state_dict(torch.load(MODEL_B_FILE, map_location=DEVICE)["model_state_dict"])
model_b.eval()


@torch.no_grad()
def predict_model_b(loader, aggregation):
    rows = []
    for batch in loader:
        images, mask = batch["images"].to(DEVICE), batch["mask"].to(DEVICE)
        mean_prob, max_prob, _ = study_forward(model_b, images, mask, USE_AMP)
        chosen = mean_prob if aggregation == "mean" else max_prob
        for i, index in enumerate(batch["indices"]):
            study = loader.dataset.studies[index]
            rows.append({
                "study_id": study["study_id"],
                "patient_id": study["patient_id"],
                "label": study["label"],
                "num_views": int(mask[i].sum()),
                "abnormal_probability": float(chosen[i]),
            })
    return pd.DataFrame(rows)


print("Model B ready (frozen checkpoint, aggregation:", policy["models"]["B"]["aggregation"] + ").")

## Reproduction check on a few validation studies

Re-scoring 24 internal-validation studies with the same code must reproduce the probabilities saved by
Notebooks 03 and 05. This confirms that the test images will go through the same preprocessing.
It runs **before** the test split is scored, so a failure stops the notebook without looking at the test.

In [ ]:
check_studies = sorted(validation_manifest["study_uid"].astype(str).unique())[:24]
check_views = validation_manifest[validation_manifest["study_uid"].astype(str).isin(check_studies)].copy()
check_paths = extract_members(check_views[PATH_COLUMN].tolist())
check_loader = DataLoader(StudyDataset(check_views, check_paths, evaluation_transform, PATH_COLUMN),
                          batch_size=8, shuffle=False, collate_fn=collate_studies)
reproduced = {
    "A": predict_model_a(check_loader).merge(val_a[["study_id", "abnormal_probability"]], on="study_id", suffixes=("", "_saved")),
    "B": predict_model_b(check_loader, policy["models"]["B"]["aggregation"]).merge(val_b[["study_id", "abnormal_probability"]], on="study_id", suffixes=("", "_saved")),
}
reproduction = {}
for name, frame in reproduced.items():
    diff = float((frame["abnormal_probability"] - frame["abnormal_probability_saved"]).abs().max())
    reproduction[name] = {"studies": int(len(frame)), "max_abs_difference": diff}
display(pd.DataFrame(reproduction).T)
for name, values in reproduction.items():
    assert values["studies"] == len(check_studies), name
    assert values["max_abs_difference"] < 2e-3, f"Model {name} does not reproduce its saved validation scores"
print("Reproduction check passed.")

## Score the locked test split (once)

In [ ]:
test_a = predict_model_a(test_loader)
test_b = predict_model_b(test_loader, policy["models"]["B"]["aggregation"])
assert len(test_a) == len(test_b) == len(test_dataset)
print("Scored test studies:", len(test_a))

## Locked-test metrics with the frozen thresholds

In [ ]:
def patient_bootstrap(frame, threshold, n_iterations=BOOTSTRAP_ITERATIONS, seed=SEED):
    patients = frame["patient_id"].astype(str).to_numpy()
    unique = np.unique(patients)
    groups = [np.flatnonzero(patients == p) for p in unique]
    y = frame["label"].to_numpy(dtype=int)
    s = frame["abnormal_probability"].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_iterations):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(unique), len(unique))])
        if np.unique(y[idx]).size < 2:
            continue
        m = classification_metrics(y[idx], s[idx], threshold)
        samples.append({"auroc": roc_auc_score(y[idx], s[idx]), "auprc": average_precision_score(y[idx], s[idx]),
                        "sensitivity": m["sensitivity"], "specificity": m["specificity"]})
    return pd.DataFrame(samples)


results, intervals = {}, []
for name, frame in (("A", test_a), ("B", test_b)):
    y = frame["label"].to_numpy(dtype=int)
    s = frame["abnormal_probability"].to_numpy(dtype=float)
    frozen = policy["models"][name]
    entry = {"auroc": float(roc_auc_score(y, s)), "auprc": float(average_precision_score(y, s))}
    entry["primary_95"] = classification_metrics(y, s, frozen["primary_threshold"])
    entry["reference_90"] = classification_metrics(y, s, frozen["reference_threshold_90"])
    # Pre-specified sensitivity analysis: drop the study holding the image duplicated from internal train.
    keep = frame["study_id"].astype(str).ne(KNOWN_DUPLICATE_TEST_STUDY).to_numpy()
    entry["primary_95_excluding_known_duplicate"] = classification_metrics(y[keep], s[keep], frozen["primary_threshold"])
    entry["auroc_excluding_known_duplicate"] = float(roc_auc_score(y[keep], s[keep]))
    entry["known_duplicate_study_in_test"] = bool((~keep).any())
    results[name] = entry
    boot = patient_bootstrap(frame, frozen["primary_threshold"])
    point = {"auroc": entry["auroc"], "auprc": entry["auprc"],
             "sensitivity": entry["primary_95"]["sensitivity"], "specificity": entry["primary_95"]["specificity"]}
    for metric, value in point.items():
        intervals.append({"model": name, "metric": metric, "point_estimate": value,
                          "lower_95_ci": float(boot[metric].quantile(0.025)),
                          "upper_95_ci": float(boot[metric].quantile(0.975)),
                          "bootstrap_samples": int(len(boot))})

summary_rows = []
for name in ("A", "B"):
    for policy_name in ("primary_95", "reference_90"):
        row = {"model": name, "policy": policy_name, "auroc": results[name]["auroc"], "auprc": results[name]["auprc"]}
        row.update(results[name][policy_name])
        summary_rows.append(row)
locked_test_table = pd.DataFrame(summary_rows)
intervals = pd.DataFrame(intervals)
display(locked_test_table[["model", "policy", "threshold", "auroc", "auprc", "sensitivity", "specificity",
                           "precision", "f1", "false_negative", "false_positive", "flagged_percent"]])
display(intervals)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name, frame in (("A", test_a), ("B", test_b)):
    fpr, tpr, _ = roc_curve(frame["label"], frame["abnormal_probability"])
    axes[0].plot(fpr, tpr, linewidth=2, label=f"Model {name} (AUROC {results[name]['auroc']:.3f})")
    m = results[name]["primary_95"]
    axes[0].scatter(1 - m["specificity"], m["sensitivity"], s=45, zorder=3)
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("Sensitivity")
axes[0].set_title("Locked test ROC (dots: frozen ≥95% thresholds)")
axes[0].legend(loc="lower right")
for axis, name in zip(axes[1:], ("A", "B")):
    m = results[name]["primary_95"]
    matrix = np.array([[m["true_negative"], m["false_positive"]], [m["false_negative"], m["true_positive"]]])
    axis.imshow(matrix, cmap="Blues")
    for r in range(2):
        for c in range(2):
            axis.text(c, r, str(matrix[r, c]), ha="center", va="center", fontsize=13)
    axis.set_xticks([0, 1], labels=["Normal", "Abnormal"])
    axis.set_yticks([0, 1], labels=["Normal", "Abnormal"])
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    axis.set_title(f"Model {name}, threshold {m['threshold']:.3f}")
plt.tight_layout()
figure_path = out_path(RUN_DIR, "locked_test_roc_and_confusion.png")
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

## Save the locked-test results

In [ ]:
test_a.assign(model="A").to_csv(out_path(RUN_DIR, "test_predictions_model_a.csv"), index=False)
test_b.assign(model="B").to_csv(out_path(RUN_DIR, "test_predictions_model_b.csv"), index=False)
locked_test_table.to_csv(out_path(RUN_DIR, "locked_test_operating_points.csv"), index=False)
intervals.to_csv(out_path(RUN_DIR, "locked_test_bootstrap_ci.csv"), index=False)
out_path(RUN_DIR, "locked_test_metrics.json").write_text(json.dumps({
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "split": "official MURA valid, XR_WRIST (held-out test, previously observed once in Phase 1)",
    "operating_policy_sha256": POLICY_SHA256,
    "test_images": int(len(test_views)),
    "test_studies": int(len(test_dataset)),
    "unreadable_images_excluded": len(unreadable),
    "patients_shared_with_development": len(overlap),
    "reproduction_check": reproduction,
    "results": results,
}, indent=2), encoding="utf-8")

a, b = results["A"], results["B"]
summary = f"""# Locked-test evaluation — Checkpoint 1

Split: official MURA `valid`, XR_WRIST — {len(test_dataset)} studies. Thresholds frozen on internal validation
(sensitivity ≥ {PRIMARY_SENSITIVITY_TARGET:.0%}) before this evaluation; policy sha256 `{POLICY_SHA256[:12]}`.

| Model | AUROC | AUPRC | Threshold | Sensitivity | Specificity | FN | FP |
|---|---:|---:|---:|---:|---:|---:|---:|
| A | {a['auroc']:.3f} | {a['auprc']:.3f} | {a['primary_95']['threshold']:.4f} | {a['primary_95']['sensitivity']:.3f} | {a['primary_95']['specificity']:.3f} | {a['primary_95']['false_negative']} | {a['primary_95']['false_positive']} |
| B | {b['auroc']:.3f} | {b['auprc']:.3f} | {b['primary_95']['threshold']:.4f} | {b['primary_95']['sensitivity']:.3f} | {b['primary_95']['specificity']:.3f} | {b['primary_95']['false_negative']} | {b['primary_95']['false_positive']} |

Pre-specified sensitivity analysis without study `{KNOWN_DUPLICATE_TEST_STUDY}` (its image is byte-identical to one
internal-train image): Model A sensitivity {a['primary_95_excluding_known_duplicate']['sensitivity']:.3f} / specificity
{a['primary_95_excluding_known_duplicate']['specificity']:.3f}; Model B sensitivity
{b['primary_95_excluding_known_duplicate']['sensitivity']:.3f} / specificity {b['primary_95_excluding_known_duplicate']['specificity']:.3f}.

Disclosure: this split was observed once in Phase 1 for a frozen reference model (reports/TEST_SET_GOVERNANCE.md).
No threshold, model or aggregation choice was changed after this evaluation. 95% CIs (patient-cluster bootstrap)
are in `locked_test_bootstrap_ci.csv`. MURA labels are study-level NORMAL/ABNORMAL, not fracture diagnoses.
"""
out_path(RUN_DIR, "locked_test_summary.md").write_text(summary.replace("\n        ", "\n"), encoding="utf-8")

for path in sorted(RUN_DIR.iterdir()):
    print("Saved:", path.name)
print()
print("POLICY_RUN_ID =", repr(RUN_DIR.name), " <- use this in Notebook 07")